In [ ]:
import pandas as pd
train_df = pd.read_csv("HeartFailure_train.csv")
test_df = pd.read_csv("HeartFailure_test.csv")

In [ ]:
### Q N°1 ###
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score
import numpy as np

train_sample = train_df.sample(frac=0.9, replace=False, random_state=0)
X_train = train_sample.drop(columns=['HeartFailure'])
y_train = train_sample['HeartFailure']
test_sample = test_df.sample(n=80, replace=False, random_state=0)
X_test = test_sample.drop(columns=['HeartFailure'])
y_test = test_sample['HeartFailure']

dt = DecisionTreeClassifier(
    criterion='entropy',
    min_impurity_decrease=0.01,
    max_depth=10,
    class_weight='balanced',
    random_state=0
)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
test_bcr = balanced_accuracy_score(y_test, y_pred)

print(f"test_bcr = {test_bcr:.4f}")

test_bcr = 0.6705


In [3]:
### Q N°2 ###
import numpy as np

# Compute n1 (positives) and n2 (negatives) in the test set
n1 = int((y_test == 1).sum())
n2 = int((y_test == 0).sum())

# Compute TPR (p1_hat) and TNR (p2_hat)
tp = int(((y_pred == 1) & (y_test == 1)).sum())
tn = int(((y_pred == 0) & (y_test == 0)).sum())
p1_hat = tp / n1   # True Positive Rate
p2_hat = tn / n2   # True Negative Rate

# Variance approximation: (1/4) * (p1*(1-p1)/n1 + p2*(1-p2)/n2)
var_bcr = 0.25 * (p1_hat * (1 - p1_hat) / n1 + p2_hat * (1 - p2_hat) / n2)
se_bcr  = np.sqrt(var_bcr)

# 95% CI using z = 1.96 (Normal approximation)
z = 1.96
CI_lower_bound = test_bcr - z * se_bcr
CI_upper_bound = test_bcr + z * se_bcr

print(f"{CI_lower_bound:.3f}, {CI_upper_bound:.3f}")

0.577, 0.764


In [4]:
### Q N°3 ###
import numpy as np
from sklearn.metrics import balanced_accuracy_score

# dt is already trained in Q1 — do NOT retrain it
test_bcrs = []

for i in range(200):
    fold       = test_df.sample(n=80, replace=False, random_state=i)
    X_fold     = fold.drop(columns=['HeartFailure'])
    y_fold     = fold['HeartFailure']
    y_pred_fold = dt.predict(X_fold)
    bcr        = balanced_accuracy_score(y_fold, y_pred_fold)
    test_bcrs.append(bcr)

mean_test_bcr = float(np.mean(test_bcrs))

print(f"mean_test_bcr = {mean_test_bcr:.4f}")
print(f"len(test_bcrs) = {len(test_bcrs)}")

mean_test_bcr = 0.6122
len(test_bcrs) = 200


In [5]:
### Q N°4 ###
import numpy as np

observed_lower_bound = float(np.percentile(test_bcrs, 2.5))
observed_upper_bound = float(np.percentile(test_bcrs, 97.5))

print(f"{observed_lower_bound:.3f}, {observed_upper_bound:.3f}")

0.508, 0.719


In [6]:
### Q N°5 ###
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score
import pandas as pd
import numpy as np

frame = pd.DataFrame(columns=[
    "indiv_test_bcr", "CI_lower_bound", "CI_upper_bound",
    "mean_test_bcr", "observed_lower_bound", "observed_upper_bound"
])

for i in range(100):

    # ── 1. Sample train (90%) and single test set (80 examples) with seed i ──
    train_sample = train_df.sample(frac=0.9, replace=False, random_state=i)
    X_train_i    = train_sample.drop(columns=['HeartFailure'])
    y_train_i    = train_sample['HeartFailure']

    test_sample  = test_df.sample(n=80, replace=False, random_state=i)
    X_test_i     = test_sample.drop(columns=['HeartFailure'])
    y_test_i     = test_sample['HeartFailure']

    # ── 2. Train DT (always random_state=0) ──
    dt_i = DecisionTreeClassifier(
        criterion='entropy',
        min_impurity_decrease=0.01,
        max_depth=10,
        class_weight='balanced',
        random_state=0
    )
    dt_i.fit(X_train_i, y_train_i)

    # ── 3. indiv_test_bcr + CI (like Q1 & Q2) ──
    y_pred_i    = dt_i.predict(X_test_i)
    indiv_bcr   = balanced_accuracy_score(y_test_i, y_pred_i)

    n1_i = int((y_test_i == 1).sum())
    n2_i = int((y_test_i == 0).sum())
    tp_i = int(((y_pred_i == 1) & (y_test_i == 1)).sum())
    tn_i = int(((y_pred_i == 0) & (y_test_i == 0)).sum())
    p1_i = tp_i / n1_i
    p2_i = tn_i / n2_i

    var_i  = 0.25 * (p1_i * (1 - p1_i) / n1_i + p2_i * (1 - p2_i) / n2_i)
    se_i   = np.sqrt(var_i)
    ci_lo  = indiv_bcr - 1.96 * se_i
    ci_hi  = indiv_bcr + 1.96 * se_i

    # ── 4. 200 test folds with seed (i+1)*j (like Q3 & Q4) ──
    bcrs_i = []
    for j in range(200):
        fold      = test_df.sample(n=80, replace=False, random_state=(i + 1) * j)
        X_fold    = fold.drop(columns=['HeartFailure'])
        y_fold    = fold['HeartFailure']
        bcr_fold  = balanced_accuracy_score(y_fold, dt_i.predict(X_fold))
        bcrs_i.append(bcr_fold)

    mean_bcr  = float(np.mean(bcrs_i))
    obs_lo    = float(np.percentile(bcrs_i, 2.5))
    obs_hi    = float(np.percentile(bcrs_i, 97.5))

    frame.loc[i] = [indiv_bcr, ci_lo, ci_hi, mean_bcr, obs_lo, obs_hi]

frame = frame.astype(float)
print(frame.head())

   indiv_test_bcr  CI_lower_bound  CI_upper_bound  mean_test_bcr  \
0        0.670494        0.577045        0.763943       0.612209   
1        0.515625        0.382629        0.648621       0.579318   
2        0.655771        0.567065        0.744477       0.627277   
3        0.647296        0.558907        0.735685       0.592165   
4        0.500000        0.364115        0.635885       0.617067   

   observed_lower_bound  observed_upper_bound  
0              0.508277              0.718685  
1              0.463092              0.692921  
2              0.538979              0.716903  
3              0.498006              0.697970  
4              0.489164              0.733781  
